# Excel → properties.js Converter

Upload your geocoded Excel file to generate a `properties.js` ready for your Refi-Map project.

**Expected columns:** Home Path, &Home, Link, Prop ID, Property Class, Property Type, Ownership Type, Description, Land Use Class, Address, City, Full Address, Vendor Company, Vendor Director, Purchaser Company, Purchaser Director, Subdivision, Site Area, Site Units, Sale Price, Sale Date, Unit Price, Unit Price Measure, Cap Rate, Total Units, Year Built, latitude, longitude

**Run each cell in order** (Shift+Enter or click ▶)

In [ ]:
# ── Step 1: Install dependencies ──────────────────────────────────────────────
!pip install -q pandas openpyxl

In [ ]:
# ── Step 2: Upload & load Excel ───────────────────────────────────────────────
import pandas as pd
from google.colab import files

print('Select your geocoded Excel file:')
uploaded = files.upload()
filename = list(uploaded.keys())[0]

df = pd.read_excel(filename)
print(f'\nLoaded {len(df)} rows, {len(df.columns)} columns')
print('Columns found:', df.columns.tolist())
df.head(3)

In [ ]:
# ── Step 3: Process all rows ──────────────────────────────────────────────────
import json
from datetime import date
from dateutil.relativedelta import relativedelta

TODAY = date.today()

# ── Helpers ──

def clean(val, fallback=''):
    try:
        if pd.isna(val): return fallback
    except (TypeError, ValueError):
        pass
    return str(val).strip()

def fmt_price(val):
    try: return '$' + format(int(float(val)), ',')
    except Exception: return 'N/A'

def fmt_ppu(price_val, measure_val):
    try:
        v = float(price_val)
        measure = clean(measure_val)
        price_str = '$' + format(int(v), ',') if v >= 1 else '$' + f'{v:.2f}'
        return price_str + '/' + measure if measure else price_str
    except Exception: return 'N/A'

def fmt_cap(val):
    try:
        v = float(val)
        if 0 < v < 1: v *= 100   # handle 0.055 → 5.5%
        return f'{v:.1f}%'
    except Exception: return '0.0%'

def fmt_date_display(dt):
    try: return pd.to_datetime(dt).strftime('%b %Y')
    except Exception: return ''

def fmt_date_iso(dt):
    try: return pd.to_datetime(dt).strftime('%Y-%m-%d')
    except Exception: return ''

def fmt_year(val):
    try:
        v = int(float(val))
        return str(v) if v > 0 else ''
    except Exception: return ''

def fmt_area(val):
    try:
        if pd.isna(val): return ''
        v = float(val)
        if v == 0: return ''
        return str(int(v)) if v == int(v) else f'{v:.4f}'.rstrip('0').rstrip('.')
    except Exception: return ''

def calc_refi(dt):
    try:
        sale = pd.to_datetime(dt).date()
        refi = sale + relativedelta(years=5)
        months_out = (refi.year - TODAY.year) * 12 + (refi.month - TODAY.month)
        is_refi = 0 <= months_out <= 18
        months_str = f'~{months_out} months' if months_out >= 0 else 'Past due'
        return refi.strftime('%b %Y'), months_str, is_refi
    except Exception: return '', '', False

# ── Main loop ──

props = []
skipped = []

for idx, row in df.iterrows():
    lat = row.get('latitude')
    lng = row.get('longitude')
    if pd.isna(lat) or pd.isna(lng):
        addr = row.get('Full Address', row.get('Address', 'unknown'))
        skipped.append('Row ' + str(idx + 2) + ': ' + str(addr) + ' — no coordinates')
        continue

    sale_dt = row.get('Sale Date', '')
    refi_date, months_out_str, is_refi = calc_refi(sale_dt)

    units_raw = row.get('Total Units')
    total_units = str(int(float(units_raw))) if pd.notna(units_raw) else '0'

    props.append({
        'address':     clean(row.get('Full Address') or row.get('Address')),
        'saleDate':    fmt_date_display(sale_dt),
        'saleDateIso': fmt_date_iso(sale_dt),
        'salePrice':   fmt_price(row.get('Sale Price')),
        'totalUnits':  total_units,
        'ppu':         fmt_ppu(row.get('Unit Price'), row.get('Unit Price Measure')),
        'capRate':     fmt_cap(row.get('Cap Rate')),
        'yearBuilt':   fmt_year(row.get('Year Built')),
        'buyer':       clean(row.get('Purchaser Company')),
        'buyerDir':    clean(row.get('Purchaser Director')),
        'seller':      clean(row.get('Vendor Company')),
        'sellerDir':   clean(row.get('Vendor Director')),
        'refiDate':    refi_date,
        'monthsOut':   months_out_str,
        'isRefi':      is_refi,
        'landUse':     clean(row.get('Land Use Class')),
        'propType':    clean(row.get('Property Type')),
        'description': clean(row.get('Description')),
        'subdivision': clean(row.get('Subdivision')),
        'siteArea':    fmt_area(row.get('Site Area')),
        'siteUnits':   clean(row.get('Site Units')),
        'propId':      str(int(float(row.get('Prop ID', 0)))),
        'lat':         round(float(lat), 8),
        'lng':         round(float(lng), 8),
    })

print('✅  ' + str(len(props)) + ' properties processed')
if skipped:
    print('⚠️   ' + str(len(skipped)) + ' rows skipped (no coordinates):')
    for s in skipped: print('   ' + s)

In [ ]:
# ── Step 4: Generate properties.js and download ───────────────────────────────

dates    = [p['saleDateIso'] for p in props if p['saleDateIso']]
date_min = min(dates) if dates else ''
date_max = max(dates) if dates else ''

def jsv(v):
    return json.dumps(v)

def jsbool(v):
    return 'true' if v else 'false'

out = [
    '// Edmonton Multifamily Sales Data',
    '// Generated: ' + str(TODAY),
    '//',
    'const DATE_MIN = ' + jsv(date_min) + ';',
    'const DATE_MAX = ' + jsv(date_max) + ';',
    '',
    'const allProps = [',
]

for p in props:
    out += [
        '  {',
        '    address: '     + jsv(p['address']) + ',',
        '    saleDate: '    + jsv(p['saleDate']) + ', saleDateIso: ' + jsv(p['saleDateIso']) + ',',
        '    salePrice: '   + jsv(p['salePrice']) + ', totalUnits: ' + jsv(p['totalUnits']) + ', ppu: ' + jsv(p['ppu']) + ', capRate: ' + jsv(p['capRate']) + ',',
        '    yearBuilt: '   + jsv(p['yearBuilt']) + ',',
        '    buyer: '       + jsv(p['buyer']) + ', buyerDir: ' + jsv(p['buyerDir']) + ',',
        '    seller: '      + jsv(p['seller']) + ', sellerDir: ' + jsv(p['sellerDir']) + ',',
        '    refiDate: '    + jsv(p['refiDate']) + ', monthsOut: ' + jsv(p['monthsOut']) + ', isRefi: ' + jsbool(p['isRefi']) + ',',
    ]
    if p['landUse']:      out.append('    landUse: '     + jsv(p['landUse']) + ',')
    if p['propType']:     out.append('    propType: '    + jsv(p['propType']) + ',')
    if p['description']:  out.append('    description: ' + jsv(p['description']) + ',')
    if p['subdivision']:  out.append('    subdivision: ' + jsv(p['subdivision']) + ',')
    if p['siteArea']:     out.append('    siteArea: '    + jsv(p['siteArea']) + ', siteUnits: ' + jsv(p['siteUnits']) + ',')
    if p['propId']:       out.append('    propId: '      + jsv(p['propId']) + ',')
    out += [
        '    lat: ' + str(p['lat']) + ', lng: ' + str(p['lng']),
        '  },',
    ]

out.append('];')

content = '\n'.join(out)
with open('properties.js', 'w', encoding='utf-8') as f:
    f.write(content)

print('✅  properties.js written — ' + str(len(props)) + ' records (' + date_min + ' → ' + date_max + ')')
print('\nFirst record preview:')
start = content.index('  {')
end   = content.index('  },') + 4
print(content[start:end])

In [ ]:
# ── Step 5: Download ──────────────────────────────────────────────────────────
from google.colab import files
files.download('properties.js')
print('Download started.')
print('Save properties.js into your Refi-Map project folder and open index.html.')

## Notes

- **Cap Rate** — the notebook handles both decimal format (`0.055`) and percentage format (`5.5`) automatically.
- **Site Area** — stored as-is (Acres or Sq Ft). The map auto-converts Sq Ft to Acres when the site area filter is used.
- **Land records** — PPU is shown as `$X,XXX/Acre` or `$X.XX/Sq Ft`. The map colour scale uses only building (non-Land) PPU values so land records don't skew the gradient. Land pins appear grey on the map.
- **Refi window** — calculated as 5 years from sale date. `isRefi: true` if the refi date is within the next 18 months from when the notebook is run.
- **Skipped rows** — any row without valid `latitude`/`longitude` values is skipped and listed in the Step 3 output.
- **Re-running** — just re-run Steps 3–5 after fixing data issues; no need to re-upload the file.